# Citi Bike demand forecasting: reproducible analysis

Companion to Neal Puri's WCTR 2026 research on spatial equity in Jersey City. **This notebook reports a NEW held-out evaluation, not the original historical R² = 0.5107.** See `REPRODUCIBILITY.md` for details.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.modeling import load_station_data, run_comparison

## 1. Load the supplied station-level data

The input contains existing stations. Excluded transit hubs match the historical analysis exclusion list; see source code.

In [ ]:
stations = load_station_data(ROOT / 'data' / 'station_level.csv')
print(f'Analyzing {len(stations)} existing stations')
display(stations[['start_station_name','number_of_origins','log_population','log_income']].head())

## 2. Explore associations

These are descriptive relationships, not causal effects.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, field, title in zip(axes, ['log_population','log_income'], ['Log population','Log median income']):
    ax.scatter(stations[field], stations['number_of_origins'], alpha=.7)
    ax.set(xlabel=title, ylabel='Trip origins')
plt.tight_layout()
plt.show()

## 3. Compare models on held-out stations

A random station-level 75/25 split prevents the same station appearing in both sets. This small sample yields uncertain estimates; no evidence here establishes forecasting accuracy for an unbuilt station.

In [ ]:
scores, y_test, outputs = run_comparison(stations, random_state=42)
display(scores.round(3))

## 4. Visualize the strongest test-MAE model

In [ ]:
name = scores.iloc[0]['model']
pred = outputs[name]['predictions']
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(y_test, pred, alpha=.75)
lo=min(min(y_test),min(pred)); hi=max(max(y_test),max(pred))
ax.plot([lo,hi],[lo,hi],'--',color='gray')
ax.set(xlabel='Actual origin trips',ylabel='Predicted origin trips',title=f'{name}: held-out stations')
plt.tight_layout()
plt.show()

## 5. Historical comparison and limitations

The earlier WCTR-era notebook saved an **in-sample quadratic R² = 0.5107**; this is not the same as the held-out scores above. Reproducing that specific model requires the unavailable intermediate CSV `nlls_predictions_with_two_independent_sqrt_nogrovest330.csv`. The GIS walking-accessibility study and proposed station siting are not reproduced by these two input datasets. Results should not be used for deployment or policy without independent validation.